<a href="https://colab.research.google.com/github/AugustinBouquillard/InTweetionists/blob/main/The_InTweetionists.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install evaluate
!pip install transformers
!pip install datasets


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [6]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps

In [7]:
# Load the training data from a JSON Lines file (one JSON object per line)
# train_data = pd.read_json('train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
# train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
# kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
# kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

# Load the training data line by line to handle potential parsing issues
train_data_list = []
with open('train.jsonl', 'r') as f:
    for line in f:
        try:
            train_data_list.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line: {line.strip()}")
            print(f"Error message: {e}")

train_data = pd.DataFrame(train_data_list)
train_data = json_normalize(train_data.to_dict(orient='records'))


# Load the Kaggle test data line by line
kaggle_data_list = []
with open('kaggle_test.jsonl', 'r') as f:
    for line in f:
        try:
            kaggle_data_list.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line: {line.strip()}")
            print(f"Error message: {e}")

kaggle_data = pd.DataFrame(kaggle_data_list)
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

In [8]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
X_train['full_text'] = X_train.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the Kaggle test data
X_kaggle['full_text'] = X_kaggle.apply(lambda tweet: extract_full_text(tweet), axis=1)

In [ ]:
# -------------------------------------------------------------
# Fine-tuning DeBERTa-v3-small classifier sur tweets
# Simplifié pour rapidité
# -------------------------------------------------------------

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import evaluate
import pandas as pd

# Vérifier si GPU disponible et activer fp16
use_fp16 = torch.cuda.is_available()

# -------------------------------------------------------------
# Préparation du modèle
# -------------------------------------------------------------
model_name = ""
tokenizer = AutoTokenizer.from_pretrained(model_name)
metric = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return metric.compute(predictions=preds, references=p.label_ids)

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

# -------------------------------------------------------------
# Train/Validation split
# -------------------------------------------------------------
df_train, df_val = train_test_split(
    pd.DataFrame({"text": X_train["full_text"], "label": y_train}),
    test_size=0.1,
    stratify=y_train,
    random_state=42
)

ds_train = Dataset.from_pandas(df_train).map(tokenize_function, batched=True)
ds_val   = Dataset.from_pandas(df_val).map(tokenize_function, batched=True)


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/139422 [00:00<?, ? examples/s]

Map:   0%|          | 0/15492 [00:00<?, ? examples/s]

In [ ]:
# -------------------------------------------------------------
# Initialisation du modèle
# -------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# -------------------------------------------------------------
# Arguments d'entraînement simplifiés
# -------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="albert_small_final",
    report_to="none",   # <-- disables wandb
    learning_rate=5e-5,      # plus sûr pour DeBERTa
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs_final",
    fp16=True,               # activer FP16
    save_total_limit=1,
)

model.gradient_checkpointing_enable()  # si VRAM limitée

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# -------------------------------------------------------------
# Lancer l'entraînement
# -------------------------------------------------------------
print("🚀 Starting training...")
trainer.train()
print("✅ Training complete.")

# -------------------------------------------------------------
# Sauvegarde du modèle + tokenizer
# -------------------------------------------------------------
trainer.save_model("deberta_small_final")
tokenizer.save_pretrained("deberta_small_final")

print("📦 Modèle sauvegardé dans 'deberta_small_final'")


pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/6_/wjb9ddn965xckzzthm1x7rnr0000gn/T/ipykernel_25509/3922914342.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


model.safetensors:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

ValueError: fp16 mixed precision requires a GPU (not 'mps').

In [ ]:
from torch.utils.data import Subset
import random

# Choisir un échantillon aléatoire
sample_size = 1000
indices = random.sample(range(len(ds_train)), sample_size)
train_sample = Subset(ds_train, indices)

# Évaluer le modèle
metrics = trainer.evaluate(eval_dataset=train_sample)
print("Accuracy sur l'échantillon random du jeu d'entraînement :", metrics["eval_accuracy"])


In [ ]:
# -------------------------------------------------------------
# Évaluation sur le test set (Kaggle)
# -------------------------------------------------------------
from datasets import Dataset
import pandas as pd
import numpy as np

# Keep all columns so we can use challenge_id later
X_kaggle_ds = Dataset.from_pandas(X_kaggle)

# Tokenize the text
X_kaggle_ds = X_kaggle_ds.map(
    tokenize_function,
    batched=True,
    remove_columns=[col for col in X_kaggle_ds.column_names if col != "challenge_id"]  # keep challenge_id
)

# Make predictions
preds = trainer.predict(X_kaggle_ds).predictions
y_pred_test = np.argmax(preds, axis=1)

# -------------------------------------------------------------
# Sauvegarde des prédictions
# -------------------------------------------------------------
output = pd.DataFrame({
    'ID': X_kaggle_ds['challenge_id'],
    'Prediction': y_pred_test
})
output.to_csv('deberta_v3_small_predictions.csv', index=False)

print("\n✅ Predictions saved to 'deberta_v3_small_predictions.csv'")
